In [2]:
import numpy as np

# 1. SETUP PROBLEM DATA (5 Delivery Stops with coordinates X, Y)
STOPS = np.array([
    [0, 0],    # Depot / Start Point
    [2, 4],    # Stop 1
    [5, 2],    # Stop 2
    [1, 6],    # Stop 3
    [8, 3]     # Stop 4
])

def calculate_total_distance(route):
    """Fitness Function: Lower distance = Better Fitness."""
    dist = 0
    # Distance from depot to first stop
    dist += np.linalg.norm(STOPS[0] - STOPS[route[0]])
    # Distance between sequential stops
    for i in range(len(route) - 1):
        dist += np.linalg.norm(STOPS[route[i]] - STOPS[route[i+1]])
    # Distance from last stop back to depot
    dist += np.linalg.norm(STOPS[route[-1]] - STOPS[0])
    return dist

def apply_levy_flight(route):
    """Lévy Flight for Discrete Routes (Small tweaks + Big jumps)."""
    new_route = route.copy()
    n_stops = len(route)
    
    # Draw step size from Levy distribution (heavy-tailed)
    u = np.random.normal(0, 1)
    v = np.random.normal(0, 1)
    step_size = u / (abs(v)**(1 / 1.5))
    
    if abs(step_size) > 1.5:
        # BIG JUMP: Reverse a whole sub-section of the route
        i, j = sorted(np.random.choice(n_stops, 2, replace=False))
        new_route[i:j] = new_route[i:j][::-1]
    else:
        # SMALL STEP: Swap two adjacent stops
        i = np.random.randint(0, n_stops - 1)
        new_route[i], new_route[i+1] = new_route[i+1], new_route[i]
        
    return new_route

def cuckoo_search_routing(n=10, Pa=0.25, Maxt=100):
    n_stops = len(STOPS) - 1  # Excluding depot (index 0)
    
    # Step 1: Initialize random routes (Host Nests)
    nests = [np.random.permutation(np.arange(1, n_stops + 1)) for _ in range(n)]
    fitness = [calculate_total_distance(r) for r in nests]
    
    # Track global best
    best_idx = np.argmin(fitness)
    best_route = nests[best_idx].copy()
    best_distance = fitness[best_idx]
    
    for t in range(Maxt):
        
        # Step 2: Generate new route via Levy Flight
        random_idx = np.random.randint(0, n)
        cuckoo_route = apply_levy_flight(nests[random_idx])
        cuckoo_dist = calculate_total_distance(cuckoo_route)
        
        # Step 3: Greedy Selection (Replace target if cuckoo is better)
        target_idx = np.random.randint(0, n)
        if cuckoo_dist < fitness[target_idx]:
            nests[target_idx] = cuckoo_route
            fitness[target_idx] = cuckoo_dist
            
        # Step 4: Abandon worst fraction (Pa) of routes
        n_abandon = int(Pa * n)
        worst_indices = np.argsort(fitness)[-n_abandon:]  # Find worst routes
        
        for idx in worst_indices:
            # Rebuild abandoned route with a fresh Levy mutation
            nests[idx] = apply_levy_flight(best_route)
            fitness[idx] = calculate_total_distance(nests[idx])
            
        # Update overall best route
        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < best_distance:
            best_distance = fitness[current_best_idx]
            best_route = nests[current_best_idx].copy()
            
    return best_route, best_distance

# --- EXECUTION ---
best_path, min_dist = cuckoo_search_routing(n=10, Pa=0.25, Maxt=200)

print("--- OPTIMIZED DELIVERY ROUTE ---")
print(f"Optimal Delivery Sequence: Depot -> {' -> '.join(map(str, best_path))} -> Depot")
print(f"Total Shortest Distance: {min_dist:.2f} km")

--- OPTIMIZED DELIVERY ROUTE ---
Optimal Delivery Sequence: Depot -> 2 -> 4 -> 3 -> 1 -> Depot
Total Shortest Distance: 22.87 km
